# Surrogate Modeling with Sensitivity Constraints and Online Learning

---

This notebook demonstrates the three-stage surrogate modeling workflow introduced in the paper:

1. **Pretraining with Sensitivity Constraints (SC)** -- train an LSTM surrogate of the HBV hydrological model on pre-generated physics simulations, regularized with a Jacobian-matching loss that enforces physically consistent input-output sensitivities.

2. **Online Surrogate Update** -- fine-tune the pretrained surrogate on new data generated during differentiable model (dHBV) training, keeping the surrogate aligned with the current LSTM parameter distribution.

3. **Inference** -- run the trained surrogate on a held-out evaluation period and compute performance metrics.

### Before running

Install the framework with:
```bash
pip install -e /path/to/surrogate_example
```

The pre-computed training and evaluation datasets are included in `./data/` as zarr stores (20-basin subset; full data available on request).

---

## Setup

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
import zarr
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# sao framework components
from sao.core.calc import batchJacobian
from sao.models.criterion.awl import AutomaticWeightedLoss
from sao.models.neural_networks.tcn import EnhancedGatedTCN

# ------------------------------------------#
CONF_PATH = './conf/surrogate.yaml'
# ------------------------------------------#

with open(CONF_PATH) as f:
    cfg = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

torch.manual_seed(42)
np.random.seed(42)

os.makedirs(cfg['pretrain']['save_path'], exist_ok=True)
os.makedirs(cfg['online']['save_path'], exist_ok=True)
os.makedirs(cfg['inference']['save_path'], exist_ok=True)

### Data loader

The zarr stores contain pre-generated HBV simulations with the following arrays:

| Key | Shape | Description |
|---|---|---|
| `forcings` | `(nt, nb, 3)` | Atmospheric forcing (prcp, tmean, pet) |
| `parameters` | `(nt, nb, 12)` | Sampled HBV parameter sets |
| `target` | `(nt, nb)` | HBV-simulated runoff (mm/day) |
| `jacobians` | `(n_t, nb, nt, 12)` | Precomputed HBV Jacobians ∂runoff/∂params |
| `j_t_idx` | `(n_t,)` | Global timestep positions for the Jacobians |
| `norm_stats/target` | `(2, nb)` | Per-basin mean and std of log-transformed target |
| `norm_stats/jacobians` | `(2, 12)` | Per-parameter mean and std of Jacobians |

In [ ]:
class SurrogateDataset(Dataset):
    """Windowed batches from a pre-generated physics model zarr store.

    Each item is a (input, target, jacobian) triplet drawn from a random
    window of length `rho` over `batch_nb` randomly selected basins.

    Parameters
    ----------
    zarr_path
        Path to zarr store.
    rho
        Window length in days.
    batch_nb
        Basins per minibatch.
    n_batches
        Minibatches per epoch.
    target_norm
        'log' applies log10(sqrt(y) + 0.1); 'none' skips normalization.
    seed
        Random seed for reproducibility.
    """

    def __init__(
        self,
        zarr_path: str,
        rho: int = 365,
        batch_nb: int = 20,
        n_batches: int = 100,
        target_norm: str = 'log',
        seed: int = 42,
    ) -> None:
        z = zarr.open(zarr_path)
        self.forcings = torch.tensor(z['forcings'][:],   dtype=torch.float32)  # (nt, nb, 3)
        self.parameters = torch.tensor(z['parameters'][:], dtype=torch.float32)  # (nt, nb, 12)
        self.target = torch.tensor(z['target'][:],     dtype=torch.float32)  # (nt, nb)
        self.jacobians = torch.tensor(z['jacobians'][:],  dtype=torch.float32)  # (n_t, nb, nt, 12)

        self.nt = self.forcings.shape[0]
        self.nb = self.forcings.shape[1]
        self.rho = rho
        self.n_batches = n_batches
        self.target_norm = target_norm

        rng = np.random.default_rng(seed)
        self.t_starts = rng.integers(0, self.nt - rho, size=n_batches)
        self.b_idx    = np.array([
            rng.choice(self.nb, min(batch_nb, self.nb), replace=False)
            for _ in range(n_batches)
        ])

    def _normalize(self, y: torch.Tensor) -> torch.Tensor:
        if self.target_norm == 'log':
            return torch.log10(torch.sqrt(torch.clamp(y, min=0.0)) + 0.1)
        return y

    def _denormalize(self, y_norm: torch.Tensor) -> torch.Tensor:
        if self.target_norm == 'log':
            return (torch.pow(10.0, y_norm) - 0.1) ** 2
        return y_norm

    def __len__(self) -> int:
        return self.n_batches

    def __getitem__(self, idx: int):
        t = int(self.t_starts[idx])
        b = self.b_idx[idx]

        x_f = self.forcings[t:t + self.rho][:, b, :]  # (rho, batch_nb, 3)
        x_p = self.parameters[t:t + self.rho][:, b, :]  # (rho, batch_nb, 12)
        x = torch.cat([x_f, x_p], dim=-1)  # (rho, batch_nb, 15)

        y = self._normalize(self.target[t:t + self.rho][:, b])  # (rho, batch_nb)

        # Precomputed HBV Jacobians at the last timestep of the window.
        # Shape: (n_t_steps, batch_nb, 12)
        t_end = min(t + self.rho - 1, self.nt - 1)
        jac = self.jacobians[:, b, t_end, :]  # (n_t_steps, batch_nb, 12)

        return x, y, jac

---
## 1. Surrogate Pretraining with Sensitivity Constraints

The surrogate (Enhanced Gated TCN, G-TCN) learns to emulate HBV's mapping

$$\hat{q} = f_{\theta}(\mathbf{x}_\text{forc},\, \boldsymbol{\phi})$$

where $\mathbf{x}_\text{forc}$ are atmospheric forcings (temporal sequence) and $\boldsymbol{\phi}$ are HBV parameters (static conditioning via FiLM). The G-TCN uses multi-scale dilated causal convolutions with gated activations and dynamic feature-wise linear modulation (FiLM) to inject parameter conditioning at every layer.

The training loss combines two terms:

$$\mathcal{L} = \mathcal{L}_\text{data} + \sum_{k \in \mathcal{S}} \mathcal{L}_{\text{SC},k}$$

where
$$\mathcal{L}_\text{data} = \text{MSE}(\hat{q},\, q_\text{HBV}), \qquad \mathcal{L}_{\text{SC},k} = \text{MSE}\!\left(\frac{\partial \hat{q}}{\partial \phi_k},\, \frac{\partial q_\text{HBV}}{\partial \phi_k}\right)$$

Loss weights are learned automatically via AutomaticWeightedLoss (AWL).

In [ ]:
train_dataset = SurrogateDataset(
    zarr_path = cfg['data']['train_path'],
    rho = cfg['data']['rho'],
    batch_nb = cfg['data']['batch_nb'],
    n_batches = cfg['data']['n_batches'],
    target_norm = cfg['data']['target_norm'],
    seed = 42,
)
train_loader = DataLoader(train_dataset, batch_size=None, shuffle=True)

print(f"Training period: {train_dataset.nt} days, {train_dataset.nb} basins")
print(f"Window length: {train_dataset.rho} days | Batches/epoch: {len(train_dataset)}")

x0, y0, j0 = train_dataset[0]
print(f"  x shape: {tuple(x0.shape)}   (rho, batch_nb, n_forc+n_params)")
print(f"  y shape: {tuple(y0.shape)}   (rho, batch_nb)")
print(f"  jac shape: {tuple(j0.shape)} (n_t_steps, batch_nb, n_params)")

In [ ]:
m_cfg = cfg['model']

# G-TCN splits inputs into forcings (temporal sequence X) and
# HBV parameters (static conditioning A injected via FiLM at each layer).
surrogate = EnhancedGatedTCN(
    n_x           = m_cfg['n_forc'],     # temporal input channels (prcp, tmean, pet)
    n_a           = m_cfg['n_params'],   # static conditioning channels (HBV parameters)
    n_out         = m_cfg['n_out'],
    width         = m_cfg['width'],
    depth         = m_cfg['depth'],
    kernel_size   = m_cfg['kernel_size'],
    causal        = m_cfg['causal'],
    dropout       = m_cfg['dropout'],
    stochastic_depth = m_cfg['stochastic_depth'],
).to(device)

sc_p     = cfg['pretrain']['sc_p']   # parameter indices to constrain
n_forc   = m_cfg['n_forc']
n_losses = 1 + len(sc_p)            # data loss + one SC term per constrained param

awl       = AutomaticWeightedLoss(n_losses).to(device)
criterion = torch.nn.MSELoss()

p_cfg     = cfg['pretrain']
optimizer = torch.optim.Adam(
    list(surrogate.parameters()) + list(awl.parameters()),
    lr           = p_cfg['lr'],
    weight_decay = p_cfg['weight_decay'],
)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, **p_cfg['scheduler_params']
)

p_idx = torch.tensor(sc_p, device=device)

print(f"G-TCN surrogate: {sum(p.numel() for p in surrogate.parameters()):,} parameters")
print(f"SC parameters: indices {sc_p}  |  AWL weights: {n_losses} terms")

### 1.1 Training loop

The G-TCN takes forcings `X: (nb, rho, n_forc)` and parameters `A: (nb, n_params)` as separate inputs. For the SC loss, `A` is reconstructed with gradient-tracked copies of the constrained parameters so that `batchJacobian_AD` can compute `∂pred/∂φ_k` at selected timesteps.

In [ ]:
def forward_and_loss(
    model,
    xb: torch.Tensor,
    yb: torch.Tensor,
    jb: torch.Tensor,
    use_sc: bool,
) -> tuple:
    """Single forward pass with combined data + SC loss.

    The G-TCN interface splits inputs into:
      X  (nb, rho, n_forc)  — temporal forcing sequence
      A  (nb, n_params)     — static HBV parameter conditioning (last timestep)

    Parameters
    ----------
    xb : (rho, nb, n_forc + n_params)
    yb : (rho, nb)
    jb : (n_t, nb, n_params)
    use_sc : whether to include the SC loss

    Returns
    -------
    loss, data_loss_scalar, sc_loss_scalar
    """
    # Split concatenated input into temporal forcings and static parameters.
    # xb is time-first (rho, nb, *); G-TCN expects batch-first (nb, rho, *).
    X  = xb[:, :, :n_forc].detach().permute(1, 0, 2)  # (nb, rho, n_forc)
    pb = xb[-1, :, n_forc:].detach()                  # (nb, n_params) — last-step params

    if use_sc:
        # Enable gradients on the subset of parameters being constrained so
        # that autograd can compute d(pred)/d(A_sel) for the SC loss.
        pb_sel = pb[:, p_idx].clone().requires_grad_(True)  # (nb, len(sc_p))

        mask = torch.ones(pb.shape[1], dtype=torch.bool, device=device)
        mask[p_idx] = False
        A_in = torch.zeros_like(pb)
        A_in[:, p_idx] = pb_sel
        A_in[:, mask]  = pb[:, mask].detach()

        # Forward pass — graph retained so batchJacobian can differentiate.
        pred = model(X, A_in)          # (nb, rho, 1)
        pred = pred.permute(1, 0, 2)   # (rho, nb, 1)  — time-first for indexing

        n_t = jb.shape[0]
        t_positions = np.linspace(0, pred.shape[0] - 1, n_t, dtype=int)
        j_pred_list = []
        for t in t_positions:
            j = batchJacobian(
                pred[t].squeeze(-1), pb_sel, graphed=True, batchx=True
            )
            j_pred_list.append(j)
        j_pred = torch.stack(j_pred_list, dim=0).squeeze()  # (n_t, nb, len(sc_p))

        data_loss = criterion(pred.squeeze(-1), yb)
        sc_losses = [
            criterion(
                j_pred[..., i],
                torch.clamp(jb[..., sc_p[i]], min=-10.0, max=10.0),
            )
            for i in range(len(sc_p))
        ]
        loss = awl(data_loss, *sc_losses)
        sc_loss_val = sum(s.item() for s in sc_losses)
    else:
        pred = model(X, pb)            # (nb, rho, 1)
        pred = pred.permute(1, 0, 2)   # (rho, nb, 1)
        data_loss = criterion(pred.squeeze(-1), yb)
        loss = data_loss
        sc_loss_val = 0.0

    return loss, data_loss.item(), sc_loss_val


def run_epoch(
    model,
    loader: DataLoader,
    optimizer,
    use_sc: bool,
    train: bool = True,
) -> dict:
    model.train() if train else model.eval()
    totals = {'loss': 0., 'data': 0., 'sc': 0.}

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb, jb in loader:
            xb = xb.squeeze().to(device)
            yb = yb.squeeze().to(device)
            jb = jb.squeeze().to(device)

            if train:
                optimizer.zero_grad()

            loss, dl, scl = forward_and_loss(model, xb, yb, jb, use_sc=use_sc)

            if train:
                loss.backward()
                optimizer.step()

            totals['loss'] += loss.item()
            totals['data'] += dl
            totals['sc']   += scl

    n = len(loader)
    return {k: v / n for k, v in totals.items()}

In [ ]:
p_cfg = cfg['pretrain']
n_epochs = p_cfg['epochs']
sc_interval = p_cfg['sc_interval']

history = {'loss': [], 'data': [], 'sc': []}

print(f"Pretraining surrogate for {n_epochs} epochs (SC on every {sc_interval} epoch(s))\n")

for epoch in tqdm(range(1, n_epochs + 1), desc='Pretraining'):
    use_sc = (epoch % sc_interval == 0) or (epoch == 1)
    stats  = run_epoch(surrogate, train_loader, optimizer, use_sc=use_sc, train=True)
    scheduler.step()

    for k in history:
        history[k].append(stats[k])

    tqdm.write(
        f"Epoch {epoch:3d}/{n_epochs}  "
        f"loss={stats['loss']:.5f}  "
        f"data={stats['data']:.5f}  "
        f"sc={stats['sc']:.5f}  "
        f"lr={scheduler.get_last_lr()[0]:.2e}"
    )

    if epoch % p_cfg['save_epoch'] == 0:
        ckpt_path = os.path.join(p_cfg['save_path'], f'surrogate_ep{epoch}.pt')
        torch.save(surrogate.state_dict(), ckpt_path)
        tqdm.write(f"  Checkpoint saved: {ckpt_path}")

print("\nPretraining complete.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

epochs = range(1, n_epochs + 1)

axes[0].plot(epochs, history['data'], label='Data loss', color='steelblue')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].set_title('Data Loss (MSE)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history['sc'], label='SC loss (sum)', color='firebrick')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE')
axes[1].set_title('Sensitivity Constraint Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Surrogate Pretraining Loss Curves', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(p_cfg['save_path'], 'pretrain_loss.png'), dpi=150)
plt.show()
print(f"AWL weights after training: {awl.params.data.detach().cpu().numpy()}")

---
## 2. Online Surrogate Update

During differentiable model (dHBV) training, the LSTM parameter predictor shifts its output distribution over successive epochs. The surrogate is therefore periodically fine-tuned on fresh simulation data produced by the current dHBV state.

In full operation this data is generated online by:
1. Forwarding a mini-batch of data through dHBV (LSTM → HBV).
2. Computing Jacobians of HBV outputs w.r.t. the new parameters.
3. Appending the new `(forcings, params, runoff, jacobians)` quadruple to a rolling replay buffer.
4. Fine-tuning the surrogate on the updated buffer every `ft_interval` epochs.

For this standalone demonstration, step 1-3 are simulated by sampling a held-out portion of the pre-computed zarr store, mimicking the distribution shift that occurs during online training.

In [ ]:
# Simulate the online buffer: load training data with a different random seed
# so that different windows and basins are selected, representing fresh
# physics model outputs from the current dHBV training iteration.
online_dataset = SurrogateDataset(
    zarr_path = cfg['data']['train_path'],
    rho = cfg['data']['rho'],
    batch_nb = cfg['data']['batch_nb'],
    n_batches = cfg['data']['n_batches'],
    target_norm = cfg['data']['target_norm'],
    seed = 999,  # different seed → new sample set (simulates parameter shift)
)
online_loader = DataLoader(online_dataset, batch_size=None, shuffle=True)

print("Online buffer ready.")
print(f"Buffer size: {len(online_dataset)} minibatches  |  "
      f"Window length: {online_dataset.rho} days")

In [ ]:
ft_epochs = cfg['online']['ft_epochs']

# Separate optimizer for fine-tuning (lower lr for stability)
ft_optimizer = torch.optim.Adam(
    list(surrogate.parameters()) + list(awl.parameters()),
    lr=p_cfg['lr'] * 0.1,
    weight_decay=p_cfg['weight_decay'],
)

ft_history = {'loss': [], 'data': [], 'sc': []}

print(f"Fine-tuning surrogate on online buffer for {ft_epochs} epochs\n")

for epoch in tqdm(range(1, ft_epochs + 1), desc='Online fine-tune'):
    stats = run_epoch(surrogate, online_loader, ft_optimizer, use_sc=True, train=True)
    for k in ft_history:
        ft_history[k].append(stats[k])
    tqdm.write(
        f"FT Epoch {epoch}/{ft_epochs}  "
        f"loss={stats['loss']:.5f}  "
        f"data={stats['data']:.5f}  "
        f"sc={stats['sc']:.5f}"
    )

ft_ckpt = os.path.join(cfg['online']['save_path'], 'surrogate_ft.pt')
torch.save(surrogate.state_dict(), ft_ckpt)
print(f"\nFine-tuned model saved: {ft_ckpt}")

---
## 3. Inference and Evaluation

Run the surrogate on the held-out evaluation period (1989–1999) and compare predictions against the reference HBV outputs.

Metrics reported:
- **NSE** -- Nash–Sutcliffe efficiency (higher is better; 1 = perfect)
- **KGE** -- Kling–Gupta efficiency
- **RMSE** -- root mean squared error
- **Bias** -- mean prediction bias

In [ ]:
z_eval   = zarr.open(cfg['data']['eval_path'])
x_f_eval = torch.tensor(z_eval['forcings'][:],   dtype=torch.float32)  # (nt_eval, nb, 3)
x_p_eval = torch.tensor(z_eval['parameters'][:], dtype=torch.float32)  # (nt_eval, nb, 12)
y_eval   = torch.tensor(z_eval['target'][:],     dtype=torch.float32)  # (nt_eval, nb)

nt_eval, nb_eval, _ = x_f_eval.shape
print(f"Evaluation period: {nt_eval} days, {nb_eval} basins")

# G-TCN interface: X (nb, nt, n_forc) and A (nb, n_params).
# Parameters are treated as static basin attributes; use the mid-period
# parameter set as the conditioning vector.
X_eval = x_f_eval.permute(1, 0, 2).to(device)        # (nb, nt_eval, n_forc)
A_eval = x_p_eval[nt_eval // 2].to(device)            # (nb, n_params)

surrogate.eval()
with torch.no_grad():
    pred_norm = surrogate(X_eval, A_eval)              # (nb, nt_eval, 1)
    pred_norm = pred_norm.squeeze(-1).permute(1, 0)    # (nt_eval, nb)

# Denormalize: invert log10(sqrt(y) + 0.1)
pred   = (torch.pow(10.0, pred_norm.cpu()) - 0.1) ** 2  # (nt_eval, nb)
target = y_eval                                           # physical units (mm/day)

print("Surrogate inference complete.")
print(f"  pred   range: [{pred.min():.4f}, {pred.max():.4f}] mm/day")
print(f"  target range: [{target.min():.4f}, {target.max():.4f}] mm/day")

In [ ]:
def nse(pred: np.ndarray, obs: np.ndarray) -> np.ndarray:
    """Nash-Sutcliffe efficiency per basin."""
    obs_mean = obs.mean(axis=0, keepdims=True)
    num = ((pred - obs) ** 2).sum(axis=0)
    den = ((obs - obs_mean) ** 2).sum(axis=0)
    return 1.0 - num / (den + 1e-8)


def kge(pred: np.ndarray, obs: np.ndarray) -> np.ndarray:
    """Kling-Gupta efficiency per basin."""
    r  = np.array([np.corrcoef(pred[:, i], obs[:, i])[0, 1] for i in range(pred.shape[1])])
    alpha = pred.std(axis=0) / (obs.std(axis=0) + 1e-8)
    beta  = pred.mean(axis=0) / (obs.mean(axis=0) + 1e-8)
    return 1.0 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)


p_np = pred.numpy()  # (nt_eval, nb)
t_np = target.numpy()  # (nt_eval, nb)

nse_vals  = nse(p_np, t_np)
kge_vals  = kge(p_np, t_np)
rmse_vals = np.sqrt(((p_np - t_np) ** 2).mean(axis=0))
bias_vals = (p_np - t_np).mean(axis=0)

print("Evaluation metrics (median across basins)")
print("-" * 40)
print(f"  NSE  : {np.median(nse_vals):.4f}")
print(f"  KGE  : {np.median(kge_vals):.4f}")
print(f"  RMSE : {np.median(rmse_vals):.4f} mm/day")
print(f"  Bias : {np.median(bias_vals):.4f} mm/day")

np.save(os.path.join(cfg['inference']['save_path'], 'pred.npy'), p_np)
np.save(os.path.join(cfg['inference']['save_path'], 'target.npy'), t_np)
print(f"\nOutputs saved to {cfg['inference']['save_path']}")

In [ ]:
import pandas as pd

# ------------------------------------------#
BASIN_IDX = 0  # index of basin to plot
PLOT_START = 365  # skip first year (warm-up)
PLOT_END = 730  # two years of predictions
# ------------------------------------------#

timesteps = pd.date_range('1989-10-01', periods=nt_eval, freq='D')
t_plot    = timesteps[PLOT_START:PLOT_END]

fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(t_plot, t_np[PLOT_START:PLOT_END, BASIN_IDX],
        label='HBV reference', color='black', linewidth=1.2, alpha=0.85)
ax.plot(t_plot, p_np[PLOT_START:PLOT_END, BASIN_IDX],
        label='Surrogate', color='steelblue', linewidth=1.0, alpha=0.9)

ax.set_xlabel('Date')
ax.set_ylabel('Runoff (mm day$^{-1}$)')
ax.set_title(
    f'Surrogate vs HBV — Basin {BASIN_IDX}  '
    f'(NSE = {nse_vals[BASIN_IDX]:.3f}, KGE = {kge_vals[BASIN_IDX]:.3f})'
)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    os.path.join(cfg['inference']['save_path'], f'hydrograph_basin{BASIN_IDX}.png'),
    dpi=150,
)
plt.show()

---
## Summary

This notebook demonstrated the full surrogate modeling workflow using the Enhanced Gated TCN (G-TCN):

| Stage | Key component | Where implemented |
|---|---|---|
| **Pretrain** | SC loss via `batchJacobian_AD` on `A` | `run_epoch` → `forward_and_loss` |
| **Online update** | Buffer replacement + fine-tune | `online_loader` + `run_epoch` |
| **Inference** | G-TCN forward with `(X, A)` split | Inference cell above |

The G-TCN separates temporal forcings (`X`) from static HBV parameter conditioning (`A`), enabling efficient FiLM-based modulation at every layer. The SC loss directly differentiates the surrogate output with respect to `A`, matching the precomputed HBV Jacobians.

For full-scale runs with CAMELS forcing data and the complete 531-basin dataset, use the CLI:

```bash
# Pretrain surrogate
python -m sao --config-name <your_config> mode=pretrain_sm

# Train dHBV with online surrogate updates
python -m sao --config-name <your_config> mode=train
```

Refer to `README.md` for configuration details.